# Lab 08-07 · Counters, Shift Registers and LFSRs

Companion to chapter [08-07](../../chapters/08-07-registers-counters-shift-registers.md). Handout: [README.md](README.md).

Every circuit here is simulated from its next-state rule, one clock at a time, as the hardware would compute it. Fibonacci LFSRs follow the chapter's convention: stages $Q_1 \ldots Q_n$ shift toward $Q_n$, the polynomial $x^n + x^k + 1$ means $Q_1^{\text{next}} = Q_n \oplus Q_k$, and the output is $Q_n$. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def popcount(x):
    return bin(x).count("1")


def lfsr_step(state, n, k):
    # bit i-1 of state holds Q_i
    fb = ((state >> (n - 1)) ^ (state >> (k - 1))) & 1
    return ((state << 1) | fb) & ((1 << n) - 1)


def lfsr_cycles(n, k):
    # lengths of the cycles into which the 2^n - 1 nonzero states fall
    seen, cycles = set(), []
    for start in range(1, 2 ** n):
        if start in seen:
            continue
        s, length = start, 0
        while True:
            s = lfsr_step(s, n, k)
            seen.add(s)
            length += 1
            if s == start:
                break
        cycles.append(length)
    return sorted(cycles)


def lfsr_bits(n, k, count, state=1):
    out = []
    for _ in range(count):
        out.append((state >> (n - 1)) & 1)
        state = lfsr_step(state, n, k)
    return out

## Part 1 · Four 4-bit counters and their switching activity

Every flip-flop that toggles charges or discharges its output node, so the number of toggles per count sets the counter's data switching power (chapter §3.7).

**🔮 Predict:** averaged over a full cycle, how many bits toggle per step in a 4-bit binary counter, a 4-bit Gray-code counter, a 4-bit ring counter and a 4-bit Johnson counter?

In [2]:
binary = list(range(16))
gray = [b ^ (b >> 1) for b in binary]


def orbit(f, start):
    s, out = start, []
    while True:
        out.append(s)
        s = f(s)
        if s == start:
            return out


ring = orbit(lambda q: (q >> 1) | ((q & 1) << 3), 0b1000)              # rotate right
johnson = orbit(lambda q: (q >> 1) | ((1 - (q & 1)) << 3), 0b0000)     # complement fed back


def toggles_per_step(seq):
    # the sequence is cyclic: include the step from its last state back to its first
    return sum(popcount(a ^ b) for a, b in zip(seq, seq[1:] + seq[:1])) / len(seq)


rates = {name: toggles_per_step(seq) for name, seq in
         (("binary", binary), ("Gray", gray), ("ring", ring), ("Johnson", johnson))}
for name, seq in (("binary", binary), ("Gray", gray), ("ring", ring), ("Johnson", johnson)):
    print(f"{name:8s} {len(seq):2d} states  {rates[name]:.3f} toggles per step  "
          + " ".join(format(s, "04b") for s in seq[:8]) + (" ..." if len(seq) > 8 else ""))
assert rates == {"binary": 1.875, "Gray": 1.0, "ring": 2.0, "Johnson": 1.0}

binary   16 states  1.875 toggles per step  0000 0001 0010 0011 0100 0101 0110 0111 ...
Gray     16 states  1.000 toggles per step  0000 0001 0011 0010 0110 0111 0101 0100 ...
ring      4 states  2.000 toggles per step  1000 0100 0010 0001
Johnson   8 states  1.000 toggles per step  0000 1000 1100 1110 1111 0111 0011 0001


The binary counter averages $(16 + 8 + 4 + 2)/16 = 1.875$ toggles per count; for $n$ bits the average approaches 2. Gray and Johnson counters change exactly one bit per step, by design. The ring counter changes two (a 1 leaves one flip-flop and enters the next), but it also needs $n$ flip-flops for only $n$ states: switching activity is only one part of the power bill, and clocking every flip-flop on every edge is the other.

In [3]:
fig, ax = plt.subplots(figsize=(5.5, 3.2))
names = list(rates)
ax.bar(names, [rates[n] for n in names], color=["#0072B2", "#009E73", "#D55E00", "#CC79A7"])
for i, n in enumerate(names):
    ax.text(i, rates[n] + 0.05, f"{rates[n]:.3f}", ha="center")
ax.set_ylabel("bits toggling per step")
ax.set_ylim(0, 2.4)
ax.set_title("Switching activity of 4-bit counters")
plt.show()

## Part 2 · Which two-tap LFSRs are maximal? (the chapter's guided exercise)

There are $n - 1$ two-tap polynomials $x^n + x^k + 1$ ($1 \le k < n$) for each $n$, so 3 + 4 + 5 + 6 + 7 = 25 for $n$ = 4 to 8.

**🔮 Predict:** roughly what fraction of the 25 give the maximal period $2^n - 1$?

The cell below steps every LFSR through every nonzero state and records its cycle lengths. It then checks each result against an independent algebraic test: a polynomial of degree $n$ is primitive if $x^{2^n} \equiv x$ modulo it, $\gcd\!\left(x^{2^{n/q}} - x,\ p\right) = 1$ for every prime $q$ dividing $n$ (together, these say it is irreducible), and $x^{(2^n - 1)/q} \not\equiv 1$ for every prime $q$ dividing $2^n - 1$ (so the powers of $x$ run through all $2^n - 1$ nonzero residues). Polynomials over GF(2) are stored as integers, bit $i$ holding the coefficient of $x^i$.

In [4]:
def pmulmod(a, b, m):
    r, dm = 0, m.bit_length() - 1
    while b:
        if b & 1:
            r ^= a
        b >>= 1
        a <<= 1
        if (a >> dm) & 1:
            a ^= m
    return r


def ppowmod(a, e, m):
    r = 1
    while e:
        if e & 1:
            r = pmulmod(r, a, m)
        a = pmulmod(a, a, m)
        e >>= 1
    return r


def pgcd(a, b):
    while b:
        while a and a.bit_length() >= b.bit_length():
            a ^= b << (a.bit_length() - b.bit_length())
        a, b = b, a
    return a


def prime_factors(n):
    f, d = [], 2
    while d * d <= n:
        if n % d == 0:
            f.append(d)
            while n % d == 0:
                n //= d
        d += 1
    return f + ([n] if n > 1 else [])


def is_primitive(p):
    n, x = p.bit_length() - 1, 2
    if ppowmod(x, 2 ** n, p) != x:
        return False
    if any(pgcd(ppowmod(x, 2 ** (n // q), p) ^ x, p) != 1 for q in prime_factors(n)):
        return False
    return all(ppowmod(x, (2 ** n - 1) // q, p) != 1 for q in prime_factors(2 ** n - 1))


tested = maximal = 0
for n in range(4, 9):
    for k in range(1, n):
        cyc = lfsr_cycles(n, k)
        is_max = cyc == [2 ** n - 1]
        assert is_max == is_primitive((1 << n) | (1 << k) | 1)
        tested += 1
        maximal += is_max
        print(f"x^{n} + x^{k} + 1:  {'MAXIMAL' if is_max else 'cycles ' + str(cyc)}")
print(f"\n{maximal} of {tested} two-tap polynomials give maximal length")
assert (tested, maximal) == (25, 10)

x^4 + x^1 + 1:  MAXIMAL
x^4 + x^2 + 1:  cycles [3, 6, 6]
x^4 + x^3 + 1:  MAXIMAL
x^5 + x^1 + 1:  cycles [3, 7, 21]
x^5 + x^2 + 1:  MAXIMAL
x^5 + x^3 + 1:  MAXIMAL
x^5 + x^4 + 1:  cycles [3, 7, 21]
x^6 + x^1 + 1:  MAXIMAL
x^6 + x^2 + 1:  cycles [7, 14, 14, 14, 14]
x^6 + x^3 + 1:  cycles [9, 9, 9, 9, 9, 9, 9]
x^6 + x^4 + 1:  cycles [7, 14, 14, 14, 14]
x^6 + x^5 + 1:  MAXIMAL
x^7 + x^1 + 1:  MAXIMAL
x^7 + x^2 + 1:  cycles [3, 31, 93]
x^7 + x^3 + 1:  MAXIMAL
x^7 + x^4 + 1:  MAXIMAL
x^7 + x^5 + 1:  cycles [3, 31, 93]
x^7 + x^6 + 1:  MAXIMAL
x^8 + x^1 + 1:  cycles [3, 63, 63, 63, 63]
x^8 + x^2 + 1:  cycles [15, 30, 30, 30, 30, 30, 30, 30, 30]
x^8 + x^3 + 1:  cycles [7, 31, 217]
x^8 + x^4 + 1:  cycles [3, 6, 6, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12]
x^8 + x^5 + 1:  cycles [7, 31, 217]
x^8 + x^6 + 1:  cycles [15, 30, 30, 30, 30, 30, 30, 30, 30]
x^8 + x^7 + 1:  cycles [3, 63, 63, 63, 63]

10 of 25 two-tap polynomials give maximal length


Ten of 25: maximal taps are common but far from universal, which is why designers take them from tables rather than guess. Two patterns are visible. Taps come in mirror pairs: if $x^n + x^k + 1$ is maximal, so is $x^n + x^{n-k} + 1$ (the same register run backward). And **$n = 8$ has none**: every trinomial of degree 8 factors. When both exponents are even the polynomial is a perfect square ($x^8 + x^2 + 1 = (x^4 + x + 1)^2$, for instance); for odd $k$ the reason is a theorem of R. G. Swan (1962), who showed that a trinomial whose degree is a multiple of 8 always has an even number of irreducible factors, so it cannot be irreducible. An 8-bit maximal LFSR therefore needs four taps, such as $x^8 + x^6 + x^5 + x^4 + 1$, which the next cell checks.

In [5]:
def is_irreducible(p):
    # the first two conditions of is_primitive
    n, x = p.bit_length() - 1, 2
    return ppowmod(x, 2 ** n, p) == x and all(pgcd(ppowmod(x, 2 ** (n // q), p) ^ x, p) == 1
                                              for q in prime_factors(n))


irreducible8 = [k for k in range(1, 8) if is_irreducible((1 << 8) | (1 << k) | 1)]
p8 = (1 << 8) | (1 << 6) | (1 << 5) | (1 << 4) | 1
print("degree-8 trinomials x^8 + x^k + 1 that are irreducible, k =", irreducible8)


def period_taps(n, taps):
    # Fibonacci LFSR with feedback Q_1 <- XOR of the tapped stages, stepped from Q_1 = 1
    s, count = 1, 0
    while True:
        fb = 0
        for t in taps:
            fb ^= (s >> (t - 1)) & 1
        s = ((s << 1) | fb) & ((1 << n) - 1)
        count += 1
        if s == 1:
            return count


print("x^8 + x^6 + x^5 + x^4 + 1 primitive:", is_primitive(p8), "  period by stepping:", period_taps(8, (8, 6, 5, 4)))
assert irreducible8 == [] and is_primitive(p8) and period_taps(8, (8, 6, 5, 4)) == 255

degree-8 trinomials x^8 + x^k + 1 that are irreducible, k = []
x^8 + x^6 + x^5 + x^4 + 1 primitive: True   period by stepping: 255


## Part 3 · What an m-sequence looks like

**🔮 Predict:** in one period of PRBS7 ($N = 127$ bits), how many 1s are there, how many runs, and what is the correlation of the sequence with a shifted copy of itself?

In [6]:
def runs(bits):
    # runs of a periodic sequence: rotate it to start at a run boundary first
    i = next(j for j in range(len(bits)) if bits[j] != bits[j - 1])
    b = bits[i:] + bits[:i]
    out, cur, ln = [], b[0], 0
    for x in b:
        if x == cur:
            ln += 1
        else:
            out.append((cur, ln))
            cur, ln = x, 1
    out.append((cur, ln))
    return out


for name, n, k in (("PRBS7", 7, 6), ("PRBS15", 15, 14)):
    N = 2 ** n - 1
    bits = lfsr_bits(n, k, N)
    r = runs(bits)
    lengths = [ln for _, ln in r]
    print(f"{name}: period {N}, ones {sum(bits)}, zeros {N - sum(bits)}, runs {len(r)}, "
          f"length-1 runs {lengths.count(1)}, longest 1s {max(l for b, l in r if b)}, "
          f"longest 0s {max(l for b, l in r if not b)}")
    assert sum(bits) == 2 ** (n - 1) and len(r) == 2 ** (n - 1) and lengths.count(1) == 2 ** (n - 2)

b7 = np.array(lfsr_bits(7, 6, 127)) * 2 - 1           # 0/1 -> -1/+1
acf = np.array([np.dot(b7, np.roll(b7, s)) / 127 for s in range(127)])
print("autocorrelation at shift 0:", acf[0], "  at every other shift:", set(np.round(acf[1:], 6)))
assert acf[0] == 1 and np.allclose(acf[1:], -1 / 127)

PRBS7: period 127, ones 64, zeros 63, runs 64, length-1 runs 32, longest 1s 7, longest 0s 6
PRBS15: period 32767, ones 16384, zeros 16383, runs 16384, length-1 runs 8192, longest 1s 15, longest 0s 14
autocorrelation at shift 0: 1.0   at every other shift: {np.float64(-0.007874)}


In [7]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
axs[0].step(range(60), lfsr_bits(7, 6, 60), where="post")
axs[0].set_xlabel("bit index")
axs[0].set_ylabel("PRBS7 output")
axs[0].set_title("The first 60 bits of PRBS7")
axs[1].stem(range(-20, 21), [acf[s % 127] for s in range(-20, 21)])
axs[1].set_xlabel("shift (bits)")
axs[1].set_ylabel("correlation (±1 form)")
axs[1].set_title("Autocorrelation: 1 at 0, −1/127 elsewhere")
plt.tight_layout()
plt.show()

Balance, the run-length distribution (half the runs have length 1, a quarter length 2, ...) and a flat autocorrelation are exactly what a coin-flip sequence would show *on average*; an m-sequence shows them *exactly* in every period. That is what makes it a good test pattern, and also why it is useless as a secret: $2n$ consecutive bits are enough to solve for the taps and predict everything that follows.

## Part 4 · A ripple counter's transient states

Each stage of a ripple counter toggles $t_{cq}$ after the stage before it falls. **🔮 Predict:** over one full cycle of a 4-bit ripple counter (16 counts), how many wrong values does the counter show in passing?

In [8]:
TCQ = 2.0  # ns per stage


def ripple(start, n=4, tcq=TCQ):
    # stage 0 toggles tcq after the clock edge; each stage that falls 1 -> 0 toggles the next tcq later
    q, t, trace, stage = start, 0.0, [], 0
    while stage < n:
        t += tcq
        old = (q >> stage) & 1
        q ^= 1 << stage
        trace.append((t, q))
        if old == 0:
            break
        stage += 1
    return trace


transients, worst = 0, 0.0
for c in range(16):
    tr = ripple(c)
    assert tr[-1][1] == (c + 1) % 16
    transients += len(tr) - 1
    worst = max(worst, tr[-1][0])
    if len(tr) > 1:
        print(f"{c:2d} -> {(c + 1) % 16:2d}: passes through", [s for _, s in tr[:-1]])
print("transient states per cycle:", transients, "   worst settling time:", worst, "ns")
assert transients == 14 and worst == 8.0

 1 ->  2: passes through [0]
 3 ->  4: passes through [2, 0]
 5 ->  6: passes through [4]
 7 ->  8: passes through [6, 4, 0]
 9 -> 10: passes through [8]
11 -> 12: passes through [10, 8]
13 -> 14: passes through [12]
15 ->  0: passes through [14, 12, 8]
transient states per cycle: 14    worst settling time: 8.0 ns


In [9]:
period = 20.0
t, val = [0.0], [0]
for c in range(16):
    for dt, s in ripple(c):
        t.append(c * period + dt)
        val.append(s)
t.append(16 * period)
val.append(val[-1])
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.step(t, val, where="post")
ax.set_xlabel("time (ns)")
ax.set_ylabel("count read from Q3..Q0")
ax.set_title("A 4-bit ripple counter, 20 ns clock, 2 ns per stage: the spikes are transient states")
plt.show()

Fourteen transient states per 16 counts: every count that flips $m$ bits passes through $m - 1$ wrong values, and the bits flipped per cycle total 30. The worst settling time, 8 ns, comes on the counts from 7 to 8 and from 15 to 0 (Example 08-07.4). A synchronous counter has none of these, because every bit changes on the same clock edge.

**Verilog.** Now run `counters_tb.v` (`python tools/run_hdl.py` from the repository root). Its PRBS measurements should agree with Part 3, and its Johnson counter should visit the 8 states listed in Part 1.